In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

# =============================================================================
# 1. ΦΟΡΤΩΣΗ & ΚΑΘΑΡΙΣΜΟΣ (Όπως πριν)
# =============================================================================
print("--- 1. Φόρτωση & Καθαρισμός ---")
df = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='POS Data')
df2 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='Hierachy Categories & Barcodes')
df3 = pd.read_excel('POS_DATA_BAPR_2024-2025_updated (3).xlsx', sheet_name='Loyalty')

# Καθαρισμός Κατηγοριών
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = df2[col].astype(str).str.strip().str.title().replace(["Nan", "None", "Na", "", "nan"], np.nan)
df2["CustomCategory"] = df2["Category B"].copy()

# Κανόνες Custom Category
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"
to_merge_dairy = ["Γιαουρτια σε συσκευασία", "Τυροκομικα σε συσκευασία", "Γαλατα σε συσκευασία", "Βουτυρα σε συσκευασία", "Κρεμα Γαλακτος σε συσκευασία"]
df2["CustomCategory"] = df2["CustomCategory"].replace(to_merge_dairy, "Γαλακτοκομικά σε συσκευασία")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση")
df2["CustomCategory"] = df2["CustomCategory"].replace(["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη")
laundry_items = ["Υγρα Πλυντηριου", "Μαλακτικα Πλυντηριου", "Ενισχυτικα-Χρωμοπαγιδες", "Σκονη Πλυντηριου"]
mask_laundry = (df2["CustomCategory"] == "Ρούχα & Ενδυση") & (df2["Category C"].isin(laundry_items))
df2.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"
xuma_map = {"Τυροκομικα": "Γαλακτοκομικά σε συσκευασία", "Αλλαντικα": "Αλλαντικα σε συσκευασία", "Μαναβικη": "Μαναβικη σε συσκευασία", "Ξηροι Καρποι": "Αλμυρα Σνακ"}
mask_xuma = df2["Category B"] == "Χυμα"
df2.loc[mask_xuma, "CustomCategory"] = df2.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")
counts = df2["CustomCategory"].value_counts()
df2["CustomCategory"] = df2["CustomCategory"].replace(counts[counts < 10].index.tolist(), "Διαφορα")

# Καθαρισμός Συναλλαγών
df.rename(columns={"Value_": "Value"}, inplace=True)
df = df[df['Quantity'] >= 1]
df = df[df['Value'] > 0]
df['Date_'] = pd.to_datetime(df['Date_'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Value', 'Barcode', 'Basket_ID', 'Date_', 'LoyaltyCard_ID'])

# Merge
df_clean = pd.merge(df, df2[['Barcode', 'CustomCategory']], on='Barcode', how='inner')

# =============================================================================
# 2. HIERARCHICAL BASKET SEGMENTATION
# =============================================================================
print("\n--- 2. Hierarchical Basket Segmentation ---")

# Pivot & Normalize
basket_pivot = df_clean.pivot_table(index='Basket_ID', columns='CustomCategory', values='Value', aggfunc='sum').fillna(0)
basket_pct = basket_pivot.div(basket_pivot.sum(axis=1), axis=0)

# ΔΕΙΓΜΑΤΟΛΗΨΙΑ (Sampling)
# Το Hierarchical είναι O(N^2), άρα για >15.000 εγγραφές είναι πολύ αργό.
# Παίρνουμε δείγμα 10.000 καλαθιών για να βρούμε τη δομή.
sample_size = 10000
if len(basket_pct) > sample_size:
    basket_sample = basket_pct.sample(n=sample_size, random_state=42)
else:
    basket_sample = basket_pct

print(f"   Using a sample of {len(basket_sample)} baskets for Hierarchical Clustering...")

# A. Δυναμική Εύρεση K στο Δείγμα (με Silhouette)
best_k_basket = 2
best_score = -1
scores = []
k_range = range(2, 9) # Δοκιμή από 2 έως 8 ομάδες

for k in k_range:
    # Agglomerative Clustering
    hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hc.fit_predict(basket_sample)
    
    score = silhouette_score(basket_sample, labels)
    scores.append(score)
    if score > best_score:
        best_score = score
        best_k_basket = k

print(f"   --> Best K found (Hierarchical): {best_k_basket} (Silhouette: {best_score:.3f})")

# B. Εμφάνιση Δενδρογράμματος (Dendrogram)
plt.figure(figsize=(10, 5))
plt.title(f"Dendrogram (Hierarchical Structure, Cut at k={best_k_basket})")
# Χρησιμοποιούμε τη linkage matrix για το plot
Z = linkage(basket_sample, method='ward')
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10., show_contracted=True)
plt.xlabel("Cluster Size")
plt.ylabel("Distance")
plt.axhline(y=Z[-best_k_basket+1, 2], c='k', linestyle='--', label=f'Cut for k={best_k_basket}')
plt.legend()
plt.show()

# C. Τελική Εκπαίδευση στο Δείγμα
final_hc = AgglomerativeClustering(n_clusters=best_k_basket, linkage='ward')
sample_labels = final_hc.fit_predict(basket_sample)

# D. Επέκταση σε ΟΛΑ τα δεδομένα (Label Propagation με KNN)
# Χρησιμοποιούμε KNN για να αντιστοιχίσουμε τα υπόλοιπα 50k καλάθια στην πιο κοντινή ομάδα του δείγματος
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(basket_sample, sample_labels)
all_labels = knn.predict(basket_pct)

basket_pct['Basket_Cluster'] = all_labels

# E. Δυναμική Ονοματοδοσία (Naming) Missions
basket_centers = basket_pct.groupby('Basket_Cluster').mean()
mission_names = {}

for i in range(best_k_basket):
    top_cat = basket_centers.loc[i].idxmax()
    top_val = basket_centers.loc[i].max()
    name = f"Mission: {top_cat}" if top_val > 0.30 else "Mission: Mixed/Refill"
    mission_names[i] = name
    
basket_pct['Mission_Name'] = basket_pct['Basket_Cluster'].map(mission_names)
print("   Basket Missions Assigned:")
print(basket_pct['Mission_Name'].value_counts())

# Heatmap Missions
top_cats_viz = basket_pct.drop(['Basket_Cluster', 'Mission_Name'], axis=1).mean().sort_values(ascending=False).head(10).index
plt.figure(figsize=(10, 6))
sns.heatmap(basket_centers[top_cats_viz].T, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Shopping Missions Profile (Hierarchical)')
plt.xticks(ticks=np.arange(0.5, best_k_basket+0.5), labels=[mission_names[i] for i in range(best_k_basket)], rotation=45, ha='right')
plt.show()

# =============================================================================
# 3. PREPARE CUSTOMER DATA
# =============================================================================
print("\n--- 3. Προετοιμασία Customer Data ---")
basket_map = basket_pct[['Mission_Name']].reset_index()
df_clean = pd.merge(df_clean, basket_map, on='Basket_ID', how='inner')

basket_summary = df_clean.groupby('Basket_ID').agg({
    'LoyaltyCard_ID': 'first', 'Date_': 'max', 'Value': 'sum', 'Mission_Name': 'first'
}).reset_index()

mission_dummies = pd.get_dummies(basket_summary['Mission_Name'])
basket_summary = pd.concat([basket_summary, mission_dummies], axis=1)

ref_date = basket_summary['Date_'].max()
mission_cols = mission_dummies.columns.tolist()

agg_rules = {'Value': ['sum', 'count', 'mean'], 'Date_': lambda x: (ref_date - x.max()).days}
for col in mission_cols: agg_rules[col] = 'sum'

customer_agg = basket_summary.groupby('LoyaltyCard_ID').agg(agg_rules)
customer_agg.columns = ['Total_Spend', 'Total_Visits', 'Avg_Basket_Value', 'Recency'] + mission_cols
customer_agg.reset_index(inplace=True)

# Calculate Percentages (Preferences)
for col in mission_cols:
    customer_agg[col + '_Pct'] = customer_agg[col] / customer_agg['Total_Visits']

# Filter & Scale
customer_agg_clean = customer_agg[customer_agg['Total_Visits'] < 400].copy()
features = ['Total_Spend', 'Total_Visits', 'Avg_Basket_Value', 'Recency'] + [c + '_Pct' for c in mission_cols]
scaler = StandardScaler()
X_cust_scaled = scaler.fit_transform(customer_agg_clean[features])

# =============================================================================
# 4. CUSTOMER SEGMENTATION (AUTO-OPTIMIZED)
# =============================================================================
print("\n--- 4. Customer Segmentation ---")

# Δυναμική εύρεση K για Πελάτες
best_k_cust = 3
best_score_cust = -1
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labs = km.fit_predict(X_cust_scaled)
    sc = silhouette_score(X_cust_scaled, labs)
    if sc > best_score_cust:
        best_score_cust = sc
        best_k_cust = k

print(f"   --> Best K found for Customers: {best_k_cust} (Score: {best_score_cust:.3f})")

kmeans_cust = KMeans(n_clusters=best_k_cust, random_state=42, n_init=10)
customer_agg_clean['Customer_Cluster'] = kmeans_cust.fit_predict(X_cust_scaled)

# Dynamic Naming
cluster_profile = customer_agg_clean.groupby('Customer_Cluster').mean(numeric_only=True)
avg_spend = customer_agg_clean['Total_Spend'].mean()
avg_rec = customer_agg_clean['Recency'].mean()

cust_names = {}
for i in range(best_k_cust):
    prof = cluster_profile.loc[i]
    fav_mission_col = prof[[c + '_Pct' for c in mission_cols]].idxmax()
    fav_mission_name = fav_mission_col.replace('_Pct', '').replace('Mission: ', '')
    
    label = f"{fav_mission_name} Lovers"
    if prof['Total_Spend'] > avg_spend * 2: label = "VIP / High Value"
    elif prof['Recency'] > avg_rec * 1.5: label = "Lapsed / Inactive"
    elif prof[fav_mission_col] < 0.4: label = "Occasional / Mixed"
    
    cust_names[i] = label

customer_agg_clean['Segment_Label'] = customer_agg_clean['Customer_Cluster'].map(cust_names)

# =============================================================================
# 5. HEATMAPS & INTERPRETATION CARDS
# =============================================================================
print("\n--- 5. Visualizing the Link (Mission Preferences) ---")

# Heatmap: Ποια Missions προτιμά κάθε Ομάδα Πελατών
mission_preferences = customer_agg_clean.groupby('Segment_Label')[[c + '_Pct' for c in mission_cols]].mean()
mission_preferences.columns = [c.replace('_Pct', '').replace('Mission: ', '') for c in mission_preferences.columns]

plt.figure(figsize=(10, 6))
sns.heatmap(mission_preferences, annot=True, cmap='Greens', fmt='.1%')
plt.title('The Link: Customer Segments vs Shopping Missions (Hierarchical Based)')
plt.ylabel('Customer Segment')
plt.xlabel('Preferred Shopping Mission')
plt.tight_layout()
plt.show()

# Unified Heatmap (RFM + Preferences)
rfm_viz = customer_agg_clean.groupby('Segment_Label')[['Total_Spend', 'Total_Visits', 'Avg_Basket_Value', 'Recency']].mean()
rfm_viz_norm = (rfm_viz - rfm_viz.mean()) / rfm_viz.std()

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.heatmap(mission_preferences, annot=True, cmap='Greens', fmt='.0%', cbar=False)
plt.title('Preferences')

plt.subplot(1, 2, 2)
sns.heatmap(rfm_viz_norm, annot=True, cmap='RdBu_r', center=0, fmt='.2f', cbar=False)
plt.title('RFM Profile (Standardized)')
plt.yticks([])

plt.tight_layout()
plt.show()

# Detailed Cards
print("\n" + "="*60)
print(f"  CUSTOMER SEGMENTATION REPORT (Based on {best_k_basket} Hierarchical Missions)")
print("="*60)

for i in range(best_k_cust):
    name = cust_names[i]
    prof = cluster_profile.loc[i]
    
    # Get top mission stats
    fav_mis_col = prof[[c + '_Pct' for c in mission_cols]].idxmax()
    fav_mis_name = fav_mis_col.replace('_Pct', '').replace('Mission: ', '')
    fav_mis_val = prof[fav_mis_col]
    
    print(f"\n🏷️  CLUSTER {i}: {name.upper()}")
    print(f"   • Size: {len(customer_agg_clean[customer_agg_clean['Customer_Cluster']==i])} Customers")
    print(f"   • Behavior: Visits={prof['Total_Visits']:.1f}, Spend={prof['Total_Spend']:.1f}€, Recency={prof['Recency']:.0f} days")
    print(f"   • Key Motivation: {fav_mis_name} ({fav_mis_val:.0%} of their baskets)")
    print("-" * 50)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

# =============================================================================
# 3ο ΕΡΩΤΗΜΑ: MARKET BASKET ANALYSIS (ΔΙΟΡΘΩΜΕΝΟ)
# =============================================================================
print("--- 3ο Ερώτημα: Market Basket Analysis & Cross-Selling Opportunities ---")

# 1. Προετοιμασία Δεδομένων
basket_matrix = df_clean.pivot_table(
    index='Basket_ID', 
    columns='CustomCategory', 
    values='Quantity', 
    aggfunc='sum'
).fillna(0)

# Μετατροπή σε 0/1
basket_sets = basket_matrix.applymap(lambda x: 1 if x > 0 else 0)

# Αφαίρεση "Διαφορα"
if 'Διαφορα' in basket_sets.columns:
    basket_sets.drop('Διαφορα', axis=1, inplace=True)

# 2. Υπολογισμός Κανόνων
frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.1)

# --- ΔΙΟΡΘΩΣΗ: Μετατροπή frozenset σε string σωστά ---
# Χρησιμοποιούμε ', '.join για να κρατήσουμε και τα δύο προϊόντα αν είναι ζευγάρι
rules['Antecedent'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['Consequent'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Ταξινόμηση
rules = rules.sort_values(['lift', 'confidence'], ascending=[False, False])
top_rules = rules.head(10).copy()

# =============================================================================
# HEATMAP ΣΥΣΧΕΤΙΣΕΩΝ (Με pivot_table για ασφάλεια)
# =============================================================================
# Χρησιμοποιούμε pivot_table με aggfunc='max' για να αποφύγουμε το ValueError αν υπάρχουν διπλότυπα
pivot_rules = top_rules.pivot_table(index='Antecedent', columns='Consequent', values='lift', aggfunc='max')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_rules, annot=True, cmap="YlGnBu", fmt=".2f", linewidths=.5)
plt.title('Top Συνδυασμοί Προϊόντων (Lift Heatmap)', fontsize=14)
plt.xlabel('...Τότε αγοράζει και αυτό (Consequent)', fontsize=12)
plt.ylabel('Αν αγοράσει αυτό... (Antecedent)', fontsize=12)
plt.tight_layout()
plt.show()

# =============================================================================
# ΕΠΙΧΕΙΡΗΜΑΤΙΚΗ ΕΡΜΗΝΕΙΑ
# =============================================================================
print("\n" + "="*60)
print("ΣΤΡΑΤΗΓΙΚΕΣ ΠΡΟΤΑΣΕΙΣ (ACTIONABLE INSIGHTS)")
print("="*60)

for i, row in top_rules.head(5).iterrows():
    ant = row['Antecedent']
    cons = row['Consequent']
    lift = row['lift']
    
    print(f"✅ ΠΡΟΤΑΣΗ BUNDLE:")
    print(f"   • Συνδυασμός: [{ant}] + [{cons}]")
    print(f"   • Ισχύς: {lift:.1f}x μεγαλύτερη πιθανότητα αγοράς.")
    print("-" * 50)

In [ ]:
import pandas as pd
!pip install networkx matplotlib
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations

# =============================================================================
# ΕΞΥΠΝΟ ΕΡΩΤΗΜΑ: CATEGORY CO-OCCURRENCE NETWORK
# "Ποιες κατηγορίες είναι αλληλένδετες;"
# =============================================================================
print("--- Smart Question: Category Co-Occurrence Network ---")

# 1. Προετοιμασία: Λίστα κατηγοριών ανά καλάθι
# Παίρνουμε μόνο τα μοναδικά προϊόντα ανά καλάθι
basket_items = df_clean.groupby('Basket_ID')['CustomCategory'].unique()

# 2. Υπολογισμός Συνύπαρξης (Co-occurrence)
# Μετράμε πόσες φορές εμφανίζεται κάθε ζευγάρι κατηγοριών μαζί
co_occurrence = {}

for items in basket_items:
    # Αν το καλάθι έχει μόνο 1 είδος, δεν μας κάνει για ζευγάρι
    if len(items) < 2:
        continue
    
    # Βρίσκουμε όλους τους συνδυασμούς ανά 2
    for pair in combinations(sorted(items), 2):
        if pair not in co_occurrence:
            co_occurrence[pair] = 0
        co_occurrence[pair] += 1

# Μετατροπή σε DataFrame
edges_df = pd.DataFrame.from_dict(co_occurrence, orient='index', columns=['Weight']).reset_index()
edges_df[['Source', 'Target']] = pd.DataFrame(edges_df['index'].tolist(), index=edges_df.index)
edges_df.drop('index', axis=1, inplace=True)

# 3. Φιλτράρισμα: Κρατάμε μόνο τις πολύ ισχυρές συνδέσεις
# (π.χ. να έχουν εμφανιστεί μαζί τουλάχιστον 100 φορές - προσάρμοσέ το ανάλογα με τον όγκο σου)
# Θέλουμε να μην γίνει "μουντζούρα" το γράφημα
threshold = edges_df['Weight'].quantile(0.95) # Κρατάμε το top 5% των συνδέσεων
edges_filtered = edges_df[edges_df['Weight'] > threshold].copy()

# 4. Δημιουργία Γραφήματος (Network Graph)
G = nx.from_pandas_edgelist(edges_filtered, 'Source', 'Target', 'Weight')

plt.figure(figsize=(12, 10))

# Υπολογισμός θέσης κόμβων (Layout)
pos = nx.spring_layout(G, k=0.5, seed=42) # k=distance between nodes

# Μέγεθος κόμβου ανάλογα με το πόσες συνδέσεις έχει (Degree)
node_sizes = [G.degree(n) * 100 for n in G.nodes()]

# Σχεδίαση
# Κόμβοι
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='skyblue', alpha=0.9)

# Ακμές (Γραμμές) - Πάχος ανάλογα με το βάρος (πόσο συχνά εμφανίζονται μαζί)
weights = [G[u][v]['Weight'] / 50 for u,v in G.edges()] # Scale down for width
nx.draw_networkx_edges(G, pos, width=weights, alpha=0.4, edge_color='gray')

# Ετικέτες
nx.draw_networkx_labels(G, pos, font_size=10, font_family='sans-serif', font_weight='bold')

plt.title(f'Δίκτυο Συνύπαρξης Κατηγοριών (Top 5% Connections)', fontsize=15)
plt.axis('off') # Κρύβουμε άξονες
plt.tight_layout()
plt.show()

# =============================================================================
# ΕΡΜΗΝΕΙΑ
# =============================================================================
print("\n--- Network Insights ---")
print("Το γράφημα δείχνει 'συστάδες' (clusters) προϊόντων που καταναλώνονται μαζί.")
print("• Οι χοντρές γραμμές δείχνουν πολύ συχνή συνύπαρξη.")
print("• Οι κόμβοι που είναι κοντά, ανήκουν συνήθως στην ίδια 'αποστολή' (π.χ. Πρωινό).")
print("• Πρόταση: Τα προϊόντα που συνδέονται με ισχυρές γραμμές πρέπει να τοποθετούνται σε κοντινούς διαδρόμους.")

# Βρες τα top 5 ζευγάρια
print("\nTop 5 Ζευγάρια Κατηγοριών:")
top_pairs = edges_filtered.sort_values(by='Weight', ascending=False).head(5)
for idx, row in top_pairs.iterrows():
    print(f"{idx+1}. {row['Source']} <---> {row['Target']} (Βρέθηκαν μαζί {row['Weight']} φορές)")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

# =============================================================================
# ΕΡΩΤΗΜΑ 3 (Γ): PARETO ANALYSIS ΑΝΑ ΚΑΤΗΓΟΡΙΑ (CATEGORY C)
# =============================================================================
print("--- 3. Pareto Analysis ανά Κατηγορία (Category Level) ---")

# 1. Merge για να φέρουμε την κατηγορία σε κάθε συναλλαγή
# Χρησιμοποιούμε το df (Transactions) και df2 (Hierarchy)
df_merged = pd.merge(df, df2[['Barcode', 'Category C']], on='Barcode', how='inner')

# 2. Ομαδοποίηση και Άθροισμα ανά Κατηγορία (Group By Category C)
cat_sales = df_merged.groupby('Category C')['Value'].sum().reset_index()
cat_sales = cat_sales.sort_values(by='Value', ascending=False)

# 3. Υπολογισμός Cumulative Stats
cat_sales['Cumulative_Value'] = cat_sales['Value'].cumsum()
cat_sales['Total_Value'] = cat_sales['Value'].sum()
cat_sales['Cumulative_Percent'] = cat_sales['Cumulative_Value'] / cat_sales['Total_Value']

# 4. ABC Classification
def abc_classify(percentage):
    if percentage <= 0.80: return 'A'
    elif percentage <= 0.95: return 'B'
    else: return 'C'

cat_sales['Class'] = cat_sales['Cumulative_Percent'].apply(abc_classify)

# =============================================================================
# ΑΠΟΤΕΛΕΣΜΑΤΑ
# =============================================================================
class_a_cats = cat_sales[cat_sales['Class'] == 'A']
count_a = len(class_a_cats)
total_cats = len(cat_sales)

print(f"\nΣΥΝΟΨΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ (ΚΑΤΗΓΟΡΙΕΣ):")
print(f"• Συνολικές Κατηγορίες: {total_cats}")
print(f"• Κατηγορίες Class A (80% τζίρου): {count_a}")
print(f"• Ποσοστό Κατηγοριών Class A: {(count_a/total_cats)*100:.1f}%")

print("\n--- Top 10 Κατηγορίες (Οι 'Πυλώνες' του Τζίρου) ---")
print(cat_sales[['Category C', 'Value', 'Cumulative_Percent']].head(10))

# =============================================================================
# ΓΡΑΦΗΜΑ (PARETO)
# =============================================================================
plt.figure(figsize=(12, 6))

# Bar plot για τις πωλήσεις
ax1 = plt.gca()
# Δείχνουμε μόνο τις Top 20 για να διαβάζεται το γράφημα
top_plot = cat_sales.head(20)
ax1.bar(top_plot['Category C'], top_plot['Value'], color='skyblue', label='Sales (€)')
plt.xticks(rotation=45, ha='right')

# Line plot για το Cumulative %
ax2 = ax1.twinx()
ax2.plot(top_plot['Category C'], top_plot['Cumulative_Percent'], color='red', marker='o', label='Cumulative %')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Όριο 80%
ax2.axhline(y=0.80, color='green', linestyle='--', label='80% Threshold')

plt.title('Pareto Analysis: Top 20 Categories', fontsize=14)
ax1.set_xlabel('Category C')
ax1.set_ylabel('Sales Value (€)')
ax2.set_ylabel('Cumulative % of Total Sales')
plt.tight_layout()
plt.show()

# =============================================================================
# ACTIONABLE INSIGHT
# =============================================================================
# =============================================================================
# ACTIONABLE INSIGHT (ΔΙΟΡΘΩΜΕΝΟ)
# =============================================================================
print("\nΕΡΜΗΝΕΙΑ:")
print(f"Οι κορυφαίες {count_a} κατηγορίες αποτελούν τον πυρήνα της κερδοφορίας.")

# Υπολογισμός του συνολικού τζίρου ξανά για σιγουριά (ως ένας αριθμός)
total_revenue = cat_sales['Value'].sum()

# Παίρνουμε την αξία της Νο1 κατηγορίας
top_cat_name = cat_sales.iloc[0]['Category C']
top_cat_value = cat_sales.iloc[0]['Value']

# Υπολογισμός ποσοστού
top_cat_pct = (top_cat_value / total_revenue) * 100

print(f"H κατηγορία '{top_cat_name}' από μόνη της φέρνει το {top_cat_pct:.1f}% του συνολικού τζίρου!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Temporal Analysis: Πωλήσεις ανά Ημέρα της Εβδομάδας ---")

# 1. Εξαγωγή της Ημέρας από την Ημερομηνία
# Το dt.day_name() βγάζει 'Monday', 'Tuesday' κτλ.
df_clean['Day_Name'] = df_clean['Date_'].dt.day_name()

# 2. Ομαδοποίηση και Άθροισμα Τζίρου ανά Ημέρα
daily_sales = df_clean.groupby('Day_Name')['Value'].sum()

# 3. Ταξινόμηση (Πολύ σημαντικό για να μην βγουν αλφαβητικά)
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# Αναδιατάσσουμε τη σειρά βάσει του days_order
daily_sales = daily_sales.reindex(days_order)

# 4. Οπτικοποίηση
plt.figure(figsize=(10, 6))

# Χρησιμοποιούμε ελληνικά labels για το γράφημα
greek_labels = ['Δευτέρα', 'Τρίτη', 'Τετάρτη', 'Πέμπτη', 'Παρασκευή', 'Σάββατο', 'Κυριακή']

# Bar Plot
daily_sales.plot(kind='bar', color='#d62728', edgecolor='black', width=0.7)

plt.title('Συνολικός Τζίρος ανά Ημέρα της Εβδομάδας', fontsize=14)
plt.xlabel('Ημέρα', fontsize=12)
plt.ylabel('Τζίρος (€)', fontsize=12)
plt.xticks(ticks=range(7), labels=greek_labels, rotation=45) # Βάζουμε τα ελληνικά
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 5. Ερμηνεία
print("\n--- Business Insight ---")
peak_day = daily_sales.idxmax()
low_day = daily_sales.idxmin()

# Μετάφραση για το print
day_map = dict(zip(days_order, greek_labels))

print(f"• Η πιο δυνατή μέρα είναι η **{day_map.get(peak_day, peak_day)}**.")
print(f"• Η πιο ήσυχη μέρα είναι η **{day_map.get(low_day, low_day)}**.")
print("Πρόταση: Τις ημέρες αιχμής χρειαζόμαστε full προσωπικό και γεμάτα ράφια (stock-up).")
print("         Τις ήσυχες μέρες μπορούμε να κάνουμε απογραφές ή να βγάλουμε προσφορές για τόνωση κίνησης.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# SMART QUESTION: LOYALTY MATRIX (Penetration vs Frequency)
# =============================================================================
print("--- Smart Analysis: Product Loyalty Matrix ---")

# 1. Προετοιμασία Δεδομένων ανά Κατηγορία
# Θέλουμε:
# - Πόσοι μοναδικοί πελάτες αγόρασαν την κατηγορία (Penetration)
# - Πόσες φορές την αγόρασαν κατά μέσο όρο (Frequency)

# Ενώνουμε με το Loyalty ID (αγνοούμε τα ανώνυμα καλάθια για να έχουμε ιστορικότητα)
df_loyal = pd.merge(df_clean, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')

category_stats = df_loyal.groupby('CustomCategory').agg({
    'LoyaltyCard_ID': 'nunique',      # Αριθμός μοναδικών πελατών
    'Basket_ID': 'count'              # Συνολικές πωλήσεις (γραμμές)
}).reset_index()

# Μετονομασία
category_stats.columns = ['CustomCategory', 'Unique_Customers', 'Total_Transactions']

# 2. Υπολογισμός Μετρικών
total_customers = df_loyal['LoyaltyCard_ID'].nunique()

# Penetration %: Ποιο ποσοστό της πελατειακής βάσης το αγόρασε έστω μία φορά;
category_stats['Penetration'] = (category_stats['Unique_Customers'] / total_customers) * 100

# Frequency: Κατά μέσο όρο, πόσες φορές το αγόρασε ο κάθε πελάτης που το προτιμά;
category_stats['Avg_Frequency'] = category_stats['Total_Transactions'] / category_stats['Unique_Customers']

# Φιλτράρισμα: Κρατάμε κατηγορίες που έχουν αγοραστεί από τουλάχιστον το 1% των πελατών (για να μην έχουμε θόρυβο)
category_stats = category_stats[category_stats['Penetration'] > 1].copy()

# 3. Visualization: SCATTER PLOT (Quadrants)
plt.figure(figsize=(12, 8))
sns.scatterplot(data=category_stats, x='Penetration', y='Avg_Frequency', size='Total_Transactions', sizes=(50, 500), alpha=0.7, hue='CustomCategory', legend=False)

# Υπολογισμός Μέσων Όρων για να χαράξουμε τους άξονες (σταυρός)
avg_pen = category_stats['Penetration'].median()
avg_freq = category_stats['Avg_Frequency'].median()

plt.axvline(x=avg_pen, color='red', linestyle='--', alpha=0.5)
plt.axhline(y=avg_freq, color='red', linestyle='--', alpha=0.5)

# 4. Annotations (Ονόματα κατηγοριών)
# Βάζουμε ετικέτες μόνο στα outliers για να μην γίνει χαμός
for i, row in category_stats.iterrows():
    # Αν είναι πολύ δημοφιλές ή πολύ συχνό, βάλε όνομα
    if row['Penetration'] > avg_pen * 1.5 or row['Avg_Frequency'] > avg_freq * 1.2:
        plt.text(row['Penetration']+0.5, row['Avg_Frequency'], row['CustomCategory'], fontsize=9)

# 5. Quadrant Labels (Το "Έξυπνο" κομμάτι)
plt.text(x=category_stats['Penetration'].max()*0.9, y=category_stats['Avg_Frequency'].max()*0.95, s="TRUE STAPLES\n(High Pen / High Freq)", fontsize=10, color='green', fontweight='bold', ha='center')
plt.text(x=category_stats['Penetration'].min()*1.5, y=category_stats['Avg_Frequency'].max()*0.95, s="NICHE LOYALTY\n(Low Pen / High Freq)", fontsize=10, color='blue', fontweight='bold', ha='center')
plt.text(x=category_stats['Penetration'].max()*0.9, y=category_stats['Avg_Frequency'].min()*1.1, s="TRAFFIC BUILDERS\n(High Pen / Low Freq)", fontsize=10, color='orange', fontweight='bold', ha='center')
plt.text(x=category_stats['Penetration'].min()*1.5, y=category_stats['Avg_Frequency'].min()*1.1, s="FILLERS\n(Low Pen / Low Freq)", fontsize=10, color='gray', fontweight='bold', ha='center')

plt.title('Product Loyalty Matrix: Δημοφιλία (Penetration) vs. Πιστότητα (Frequency)', fontsize=14)
plt.xlabel('Πόσοι διαφορετικοί πελάτες αγοράζουν το προϊόν)', fontsize=10)
plt.ylabel('Όσοι το αγοράζουν, πόσο συχνά επιστρέφουν για αυτό;', fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

# =============================================================================
# ΕΡΜΗΝΕΙΑ ΓΙΑ ΤΗΝ ΠΑΡΟΥΣΙΑΣΗ
# =============================================================================
print("\n--- BUSINESS INSIGHTS ---")
print("Χωρίσαμε τα προϊόντα σε 4 στρατηγικές κατηγορίες:")
print("1. Staples (Πάνω Δεξιά): Τα 'βασικά' του σπιτιού. Δεν πρέπει να λείψουν ποτέ.")
print("2. Niche Loyalty (Πάνω Αριστερά): Προϊόντα που αγοράζουν λίγοι, αλλά φανατικά! Αν τα καταργήσουμε, θα χάσουμε πιστούς πελάτες.")
print("3. Traffic Builders (Κάτω Δεξιά): Προϊόντα που αγοράζουν πολλοί αλλά αραιά (π.χ. Χαρτικά/Απορρυπαντικά). Ιδανικά για φυλλάδια προσφορών.")